# SPiFiL quickstart — fit filters, then use them like any torch encoder

SPiFiL learns convolutional filters **without backpropagation**. Superpixel
centers give candidate feature points, patches around them are scored
(Fisher / Mahalanobis) and selected (rank + diversity, greedily), and the
chosen patches *become* the conv kernels, with z-score normalization folded
into the weights and bias. What comes out is a plain PyTorch `nn.Module`.

This notebook walks the whole path:

1. load a dataset and look at it,
2. inspect layer 0 — superpixels and the seeds they reduce to,
3. fit a three-layer stack,
4. see which filters were chosen and what they encode,
5. export the model and reload it from disk,
6. freeze it, attach a task head and train it, or fit the paper's linear SVM.

It runs on the datasets bundled in [`data/`](data/README.md): the
**one-image-per-class** training sets the paper's encoders were learned from,
for `eggs`, `cysts` and `larvae`, in three splits each. Point the next cell at
your own folders to use something else.

## 0. Setup

```bash
uv sync                  # or: pip install -e .
uv sync --group disf     # optional: adds DISF, the published superpixels
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from skimage.segmentation import mark_boundaries
from torch import nn

from spifil import (
    DISF,
    SLIC,
    ArchSpec,
    DiversitySelector,
    FisherScorer,
    LayerSpec,
    Learner,
    Mahalanobis,
    Medoids,
    ProgressBar,
    SpifilDataset,
    load_encoder,
    prepare_sample,
)
from spifil.superpixels.disf import require_pyift

torch.manual_seed(0)

# The bundled datasets live in examples/data; find them from wherever this
# notebook was started.
DATA = next(
    p / "examples" / "data"
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "examples" / "data").is_dir()
)

DATASET, SPLIT = "eggs", "split1"  # or "cysts" / "larvae"; split1..split3
images_dir = DATA / DATASET / SPLIT / "images"
masks_dir = DATA / DATASET / SPLIT / "masks"
print("dataset:", images_dir.relative_to(DATA.parent.parent))

## 1. The dataset

`SpifilDataset.from_folders` pairs each image with a mask of the same name and
takes the class from the filename prefix — `000002_00001142.png` is class 2.
The mask restricts where superpixels may be placed, which is what keeps every
candidate filter on the class of interest; without one the whole image is fair
game.

In [ ]:
data = SpifilDataset.from_folders(images_dir, masks_dir)
print(f"{len(data)} images, classes {data.classes}, labels {data.labels}")

n_show = min(len(data), 4)
fig, axes = plt.subplots(n_show, 2, figsize=(5, 2.6 * n_show), squeeze=False)
for row in range(n_show):
    axes[row, 0].imshow(data.load_image(row))
    axes[row, 0].set_title(f"{data[row].name} (class {data[row].label})", fontsize=9)
    mask = data.load_mask(row)
    axes[row, 1].imshow(mask, cmap="gray")
    axes[row, 1].set_title("mask", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
fig.tight_layout()

## 2. Layer 0 — color features, superpixels, seeds

Layer 0 is not learned. It converts the image to multiband features, segments
*those features* into superpixels, and reduces each region to a single
representative pixel — the **seed**. Every later stage works on patches
centered on those seeds.

Two components are swappable here, and both change the result:

- **the superpixel algorithm.** `DISF` is what the SPiFiL paper uses; it needs
  the optional `disf` extra. `SLIC` needs nothing beyond scikit-image and is
  the default, so this notebook falls back to it when DISF is unavailable.
- **the seed rule.** `Medoids` — the pixel most representative of a region's
  appearance — is the definition the paper gives, and the default.

In [ ]:
try:
    require_pyift()
    superpixels, which = DISF(), "DISF (the published choice)"
except ImportError:
    superpixels, which = SLIC(), "SLIC (DISF not installed — different filters)"
print("superpixels:", which)

n_superpixels = 50
layer0 = prepare_sample(
    data,
    0,
    superpixels=superpixels,
    seed_extractor=Medoids(),
    n_superpixels=n_superpixels,
)
print("features:", layer0.features.shape, "| seeds:", len(layer0.seeds))

image = data.load_image(0)
coords = layer0.seeds.coords.numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(mark_boundaries(image, layer0.superpixel_labels))
axes[0].set_title(f"{layer0.superpixel_labels.max()} superpixels")
axes[1].imshow(image)
axes[1].scatter(coords[:, 1], coords[:, 0], s=14, c="yellow", edgecolors="black")
axes[1].set_title(f"{len(layer0.seeds)} medoid seeds")
axes[2].imshow(layer0.features[0], cmap="gray")
axes[2].set_title("layer-0 features (band 0)")
for ax in axes:
    ax.axis("off")
fig.tight_layout()

## 3. Fit

The `Learner` owns the layer loop and nothing else — every stage is injected,
so swapping the scorer or the selector is a constructor argument.

Fitting is a handful of long steps rather than thousands of small ones: per
layer, score every seed, allocate filters per class, select patches, fold them
into kernels, encode every image, project the seeds onto the pooled grid.
There is no optimizer and no epoch loop.

In [ ]:
arch = ArchSpec(
    layers=[
        LayerSpec(kernel_size=5, out_channels=16),
        LayerSpec(kernel_size=5, out_channels=32),
        LayerSpec(kernel_size=5, out_channels=48),
    ],
    stdev_factor=0.01,
)

learn = Learner(
    data,
    arch,
    superpixels=superpixels,
    seed_extractor=Medoids(),
    metric=Mahalanobis(),
    scorer=FisherScorer(),
    selector=DiversitySelector(alpha=0.5, pool_factor=5),  # the paper's alpha, gamma
    cbs=[ProgressBar()],
    n_superpixels=n_superpixels,
)
learn.fit()

## 4. What came out

Each layer's `FilterBank` carries the filters *and* their provenance: which
class each one came from, and the z-score statistics they were folded from.
Filters are allocated per class by integer division, which is why a layer can
come out narrower than asked for: on `cysts` the requested 16/32/48 is built
as **12/30/48**, because 6 classes divide 16 and 32 with a remainder.

In [ ]:
for layer in range(1, len(arch.layers) + 1):
    bank = learn.state[layer].bank
    per_class = np.bincount(bank.labels.numpy())[1:]
    print(
        f"layer {layer}: {len(bank)} filters "
        f"(asked for {arch.layers[layer - 1].out_channels}), "
        f"per class {per_class.tolist()}, weight {tuple(bank.weight.shape)}"
    )

# The paper's encoder has 52,496 parameters (16/32/48); cysts' 12/30/48 has fewer.
n_params = sum(
    learn.state[layer].bank.weight.numel() + learn.state[layer].bank.bias.numel()
    for layer in range(1, len(arch.layers) + 1)
)
print(f"encoder parameters: {n_params:,}")

Layer 1's filters are 5x5 patches of the three color bands, so they can be
looked at directly. These are the *selected patches* — the numbers before the
z-score fold-in, which is what makes them comparable to each other.

The three bands are normalized L\*, a\*, b\*, shown as RGB for compactness.
Read them as "which local color/contrast pattern is this filter tuned to", not
as true color.

In [ ]:
selected = learn.state[1].selected
patches = selected.feats.reshape(len(selected), 3, 5, 5)  # (n, C, 5, 5)

n_tiles = min(len(selected), 16)
columns = 8
rows = -(-n_tiles // columns)
fig, axes = plt.subplots(
    rows, columns, figsize=(1.5 * columns, 1.7 * rows), squeeze=False
)
for ax in axes.ravel():
    ax.axis("off")
tiles = zip(axes.ravel(), patches[:n_tiles], selected.labels.tolist(), strict=False)
for ax, patch, label in tiles:
    tile = patch.permute(1, 2, 0).numpy()
    ax.imshow((tile - tile.min()) / np.ptp(tile).clip(1e-6), interpolation="nearest")
    ax.set_title(f"class {label}", fontsize=8)
fig.suptitle("layer-1 filters — the selected patches themselves", y=1.02)
fig.tight_layout()

And this is what the stack *encodes*. Each block is
`conv -> ReLU -> max-pool(stride 2)`, so the spatial grid halves at every
layer.

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(12, 6))
for row, layer in enumerate((1, 2, 3)):
    features = learn.state[layer].features[0]
    for column in range(6):
        axes[row, column].imshow(features[column], cmap="magma")
        axes[row, column].axis("off")
    axes[row, 0].set_title(
        f"layer {layer}: {tuple(features.shape)}", loc="left", fontsize=9
    )
fig.tight_layout()

## 5. Export and reload

`export` writes a bundle: `model.pt` (a plain `state_dict`),
`architecture.json`, and `labels{L}.txt` — the class each filter came from.

The description records the stack **as built**, not as requested, and it
records **which color transform** produced the features the filters were
fitted on. That last field is not bookkeeping: Lab, RGB and any other
three-band space are interchangeable as far as tensor shapes are concerned, so
without it a bundle can be handed the wrong preprocessing and will simply
return wrong numbers — no error, no shape mismatch.

In [ ]:
import json
import tempfile

bundle = Path(tempfile.mkdtemp()) / "model"
learn.export(bundle)

print("\n".join(sorted(p.name for p in bundle.iterdir())))
described = json.loads((bundle / "architecture.json").read_text())
print()
print(
    json.dumps({k: described[k] for k in ("format", "in_channels", "color")}, indent=2)
)
print("layer 1:", json.dumps(described["layers"][0]))

## 6. Freeze it and train a head

`load_encoder` returns a `SpifilEncoder` — the color transform *and* the
fitted stack as one module. That pairing is the point: `SpifilNet` alone eats
layer-0 features, not images, so handing a downstream model the network by
itself is handing it half an artifact.

From here it is ordinary PyTorch: freeze the encoder, put a head on it, train
the head.

In [ ]:
encoder = load_encoder(bundle).freeze()


class Head(nn.Module):
    """Global average pool -> linear. The simplest head that works."""

    def __init__(self, in_channels: int, n_classes: int) -> None:
        super().__init__()
        self.fc = nn.Linear(in_channels, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x.mean(dim=(2, 3)))


head = Head(encoder.out_channels, len(data.classes))
model = nn.Sequential(encoder, head)  # images in, logits out

images = np.stack([data.load_image(i) for i in range(len(data))])
targets = torch.tensor([data.classes.index(label) for label in data.labels])
print("images:", images.shape, "| targets:", targets.tolist())

Because the encoder is frozen its output never changes, so compute it once and
train the head on the cached features instead of re-encoding every step. The
full `nn.Sequential` still works end to end; it is just wasteful in a loop.

In [ ]:
with torch.no_grad():
    features = encoder(images)
print("encoded:", tuple(features.shape))

# The composed model and the cached path agree, as they must.
with torch.no_grad():
    assert torch.equal(model(images), head(features))

optimizer = torch.optim.Adam(head.parameters(), lr=0.05)
losses = []
for _ in range(60):
    optimizer.zero_grad()
    loss = nn.functional.cross_entropy(head(features), targets)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

accuracy = (head(features).argmax(dim=1) == targets).float().mean().item()
print(
    f"loss {losses[0]:.3f} -> {losses[-1]:.3f}"
    f" | fit on the training set: {accuracy:.0%}"
)

plt.figure(figsize=(5, 3))
plt.plot(losses)
plt.xlabel("step")
plt.ylabel("cross-entropy")
plt.title(f"head training ({len(data)} images — wiring demo, not a result)")
plt.tight_layout()

That number is the head fitting the images it was shown. It says the gradients
flow and the features carry *something*; it says nothing about generalization.
For a real evaluation, hold out a test split and train the head on the
training half only.

Worth checking explicitly: the encoder really is frozen, so all of that
learning happened in the head.

In [ ]:
print(
    "encoder params requiring grad:", sum(p.requires_grad for p in encoder.parameters())
)
print("head params requiring grad:   ", sum(p.requires_grad for p in head.parameters()))

### A linear SVM instead, as in the paper

The paper's tables skip the trained head: a linear SVM (one-vs-one, `C=100`) is fitted on the frozen layer-3 map, flattened to one vector per image: 48 x 25 x 25 = 30,000 values for these 200x200 images. Nothing in that pipeline uses gradients, the encoder or the classifier.

Fitted on the bundled images alone, the score below is again a wiring check. To evaluate as the paper does, encode the `train` and `test` lists of the pinned splits (see "The published splits" in the README), fit on the first and score on the second.

In [ ]:
from sklearn.metrics import cohen_kappa_score, f1_score
from sklearn.svm import SVC

X = features.flatten(start_dim=1).numpy()  # (n_images, C * H * W)
y = targets.numpy()
svm = SVC(kernel="linear", C=100, decision_function_shape="ovo").fit(X, y)

predicted = svm.predict(X)
print("features per image:", X.shape[1])
print(
    f"accuracy {(predicted == y).mean():.0%}"
    f" | weighted F1 {f1_score(y, predicted, average='weighted'):.3f}"
    f" | kappa {cohen_kappa_score(y, predicted):.3f}"
    " (training set: wiring demo, not a result)"
)

## 7. Other things you can do with it

`upto` is a layer number, where layer 0 is the color transform's output — so
`upto=0` is the transform alone and `upto=2` stops after the second block.
Useful for feeding a head from an earlier, higher-resolution layer.

In [ ]:
with torch.no_grad():
    for upto in range(len(arch.layers) + 1):
        print(f"upto={upto}: {tuple(encoder(images[:1], upto=upto).shape)}")

Every component is swappable at construction — `selector=TopN()`,
`metric=Euclidean()`, `scorer=DistanceSumScorer()`, `superpixels=DISF()`, a
different `ArchSpec`. From the command line the same choices are Hydra config
groups:

```bash
uv run spifil-fit data.images_dir=examples/data/eggs/split1/images \
                  data.masks_dir=examples/data/eggs/split1/masks \
                  superpixels=disf n_superpixels=50 arch=paper \
                  selector.alpha=0.5 selector.pool_factor=5
uv run spifil-fit -m selector.alpha=0.3,0.5,0.7          # a sweep
```